In [0]:
"""
Data Quality - Validity Check Task
Validates numeric and date comparisons between columns.
"""
from pyspark.sql import functions as F
import json
import re
import time

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("column_1", "")
dbutils.widgets.text("column_2", "")
dbutils.widgets.text("operator", ">")
dbutils.widgets.text("check_type", "numeric")  # numeric, date
dbutils.widgets.text("single_column", "")  # Not used anymore

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
COLUMN_1 = dbutils.widgets.get("column_1")
COLUMN_2 = dbutils.widgets.get("column_2")
OPERATOR = dbutils.widgets.get("operator")
OPERATOR = "==" if OPERATOR == "=" else OPERATOR
CHECK_TYPE = dbutils.widgets.get("check_type")

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Validity Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Check Type: {CHECK_TYPE}")

# Read input data
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

# Start with row_id only
df_checks = df_input.select("row_id")

# Only numeric and date comparisons are supported now
if not COLUMN_1 or not COLUMN_2:
    raise ValueError(f"{CHECK_TYPE} comparison check requires both column_1 and column_2")

print(f"Validation: {COLUMN_1} {OPERATOR} {COLUMN_2}")

# Map operator symbol to word
operator_word_map = {
    ">": "greater",
    ">=": "greater_equal",
    "<": "less",
    "<=": "less_equal",
    "==": "equal",
    "!=": "not_equal"
}
operator_word = operator_word_map.get(OPERATOR, OPERATOR)
check_col_name = f"{COLUMN_1}_{operator_word}_{COLUMN_2}_validity_check"
print(f"Creating validity check: {check_col_name}")

# Build condition based on operator and check type
if CHECK_TYPE == "date":
    # Convert to date type for comparison, specify format to avoid cast errors
    date_format = "dd/MM/yyyy"  # Adjust if your format is different
    col1_expr = F.to_date(F.col(COLUMN_1), date_format)
    col2_expr = F.to_date(F.col(COLUMN_2), date_format)
else:
    # Numeric comparison (use as-is)
    col1_expr = F.col(COLUMN_1)
    col2_expr = F.col(COLUMN_2)

# Map operator to condition
if OPERATOR == ">":
    condition = col1_expr > col2_expr
elif OPERATOR == ">=":
    condition = col1_expr >= col2_expr
elif OPERATOR == "<":
    condition = col1_expr < col2_expr
elif OPERATOR == "<=":
    condition = col1_expr <= col2_expr
elif OPERATOR == "==":
    condition = col1_expr == col2_expr
elif OPERATOR == "!=":
    condition = col1_expr != col2_expr
else:
    raise ValueError(f"Unsupported operator: {OPERATOR}")

# Add validity check: 1 if valid, 0 if invalid, NULL if either column is NULL
df_checks = df_checks.join(
    df_input.select("row_id", COLUMN_1, COLUMN_2).withColumn(
        check_col_name,
        F.when(
            F.col(COLUMN_1).isNull() | F.col(COLUMN_2).isNull(), 
            None
        ).otherwise(
            F.when(condition, 1).otherwise(0)
        )
    ).select("row_id", check_col_name),
    "row_id",
    "left"
)

# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

# Output table name
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_validity"

print(f"Writing to: {output_table}")

# Check if table exists to determine write mode
table_exists = spark.catalog.tableExists(output_table)
write_mode = "append" if table_exists else "overwrite"

print(f"Table exists: {table_exists}, using mode: {write_mode}")

# Write to Unity Catalog with retry logic for concurrent append exception
max_retries = 3
for attempt in range(max_retries):
    try:
        writer = df_checks.write.mode(write_mode)
        
        if write_mode == "overwrite":
            # Only set overwriteSchema when creating table for the first time
            writer = writer.option("overwriteSchema", "true")
        else:
            # Enable schema merging when appending (different check columns per validity type)
            writer = writer.option("mergeSchema", "true")
        
        writer.saveAsTable(output_table)
        print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")
        break
    except Exception as e:
        if "ConcurrentAppendException" in str(e) and attempt < max_retries - 1:
            print("Concurrent append detected, retrying...")
            time.sleep(2)
        elif "already exists" in str(e).lower() and attempt < max_retries - 1:
            print("Table was created by another task, switching to append mode...")
            write_mode = "append"
            time.sleep(2)
        else:
            raise

# Display statistics
valid_count = df_checks.filter(F.col(check_col_name) == 1).count()
invalid_count = df_checks.filter(F.col(check_col_name) == 0).count()
null_count = df_checks.filter(F.col(check_col_name).isNull()).count()
validity_pct = (valid_count / total_rows * 100) if total_rows > 0 else 0

print(f"\nValidity Statistics:")
print(f"  Validation: {COLUMN_1} {OPERATOR} {COLUMN_2}")
print(f"  Valid rows: {valid_count} ({validity_pct:.2f}%)")
print(f"  Invalid rows: {invalid_count} ({(invalid_count/total_rows*100) if total_rows > 0 else 0:.2f}%)")
print(f"  NULL rows: {null_count} ({(null_count/total_rows*100) if total_rows > 0 else 0:.2f}%)")
print(f"  Total rows: {total_rows}")